# 00 · Exploración de datos — Cloud Provider Analytics

**Minería de Datos II · ISTEA 2C 2026 · Primera entrega**

Notebook de evidencia mínima de lectura y exploración de las fuentes de Landing con PySpark.

**Alcance:** lectura, conteos, nulos, duplicados, evolución de esquema y chequeos de calidad.
No limpia ni escribe datos: el resultado alimenta el documento `docs/02_5v_e_inventario_fuentes.md`.

> Se ejecuta de arriba hacia abajo en Google Colab (o local) sin pasos manuales.
> En exploración se usa esquema **inferido** a propósito, para observar cómo llegan los datos; en Bronze se usarán esquemas **explícitos**.

## 1. Instalar PySpark

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install pyspark

## 2. Obtener los datos

Los datos de muestra viven en el repositorio (`data/cloud_provider_challenge_dataset_v1.zip`).
En Colab se clona el repo; en local se usa la carpeta del repo. El zip se descomprime en `data/raw/`
(copia de trabajo; el zip original actúa como Landing inmutable).

In [ ]:
import subprocess, zipfile
from pathlib import Path

REPO_URL = "https://github.com/damiann8n/cloud-provider-analytics.git"
ZIP_NAME = "cloud_provider_challenge_dataset_v1.zip"

if IN_COLAB:
    ROOT = Path("/content/cloud-provider-analytics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

ZIP_PATH = ROOT / "data" / ZIP_NAME
RAW_DIR = ROOT / "data" / "raw"
if not (RAW_DIR / "datalake").exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(RAW_DIR)

LANDING = RAW_DIR / "datalake" / "landing"
print("Landing:", LANDING)
for p in sorted(LANDING.iterdir()):
    n = len(list(p.glob("*.jsonl"))) if p.is_dir() else None
    print(f"  {p.name:<28} {'%d archivos' % n if n is not None else '%.1f KB' % (p.stat().st_size / 1024)}")

## 3. Sesión de Spark

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("cpa-exploracion")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

## 4. Lectura de las fuentes batch (CSV)

Se leen como texto (`inferSchema=False`) para no ocultar tipos ambiguos; los vacíos llegan como `null`.

In [ ]:
FUENTES = {
    "customers_orgs":    "org_id",
    "users":             "user_id",
    "resources":         "resource_id",
    "support_tickets":   "ticket_id",
    "marketing_touches": "touch_id",
    "nps_surveys":       None,          # sin clave propia: (org_id, survey_date)
    "billing_monthly":   "invoice_id",
}

dfs = {
    nombre: spark.read.option("header", True).option("inferSchema", False).csv(str(LANDING / f"{nombre}.csv"))
    for nombre in FUENTES
}
for nombre, df in dfs.items():
    print(f"{nombre:<18} filas={df.count():>5}  columnas={len(df.columns)}")

## 5. Perfil por fuente: nulos y duplicados por clave natural

In [ ]:
def perfil(nombre, df, clave):
    total = df.count()
    nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    distintos = df.agg(*[F.countDistinct(c).alias(c) for c in df.columns]).first().asDict()
    dup = total - df.select(clave).distinct().count() if clave else total - df.distinct().count()
    print(f"\n=== {nombre} · filas={total} · duplicados por {'clave ' + clave if clave else 'fila completa'}={dup}")
    print(f"  {'columna':<22}{'nulos':>7}{'%':>8}{'distintos':>11}")
    for c in df.columns:
        print(f"  {c:<22}{nulos[c]:>7}{100 * nulos[c] / total:>7.1f}%{distintos[c]:>11}")

for nombre, clave in FUENTES.items():
    perfil(nombre, dfs[nombre], clave)

## 6. Dominios de las columnas categóricas

In [ ]:
categoricas = {
    "customers_orgs":  ["industry", "hq_region", "plan_tier", "is_enterprise", "lifecycle_stage"],
    "resources":       ["service", "region", "state"],
    "support_tickets": ["category", "severity", "sla_breached"],
    "billing_monthly": ["month", "currency"],
}
for nombre, cols in categoricas.items():
    for c in cols:
        valores = dfs[nombre].groupBy(c).count().orderBy(F.desc("count")).collect()
        print(f"{nombre}.{c}: " + ", ".join(f"{r[c]}={r['count']}" for r in valores))

## 7. Stream de eventos de uso (JSONL)

Lectura batch del directorio completo solo para explorar. En la implementación se procesará con Structured Streaming.

In [ ]:
eventos = (
    spark.read.json(str(LANDING / "usage_events_stream"))
    .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1))
)
eventos.printSchema()
print("Eventos:", eventos.count(), "· archivos:", eventos.select("source_file").distinct().count())

> Spark infiere `value` como **string** porque en el JSON llega mezclado: número (`133.0`) o texto (`"133.0"`).
> En esta muestra todos los textos son casteables a número, pero el pipeline debe aplicar cast con *fallback* a null
> por si llegan valores no numéricos. Es la evidencia del problema de *tipos ambiguos*.

### 7.1 Evolución de esquema (v1 → v2)

In [ ]:
(eventos.groupBy("schema_version")
    .agg(
        F.count("*").alias("eventos"),
        F.min("timestamp").alias("desde"),
        F.max("timestamp").alias("hasta"),
        F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon_kg"),
        F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_genai_tokens"),
    )
    .orderBy("schema_version")
    .show(truncate=False))

### 7.2 Unicidad de `event_id`, nulos y tipos de `value`

In [ ]:
total = eventos.count()
print("event_id únicos:", eventos.select("event_id").distinct().count(), "de", total)

eventos.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in
                ["event_id", "org_id", "resource_id", "service", "metric", "value", "unit", "cost_usd_increment"]]).show()

# value llega como string: se distinguen nulos, numéricos válidos y no numéricos
(eventos
    .withColumn("tipo_value",
        F.when(F.col("value").isNull(), "null")
         .when(F.col("value").cast("double").isNotNull(), "numérico")
         .otherwise("no numérico"))
    .groupBy("tipo_value").count().show())

### 7.3 Dominios: servicios, regiones, métricas y unidades

In [ ]:
for c in ["service", "region", "metric", "unit"]:
    eventos.groupBy(c).count().orderBy(F.desc("count")).show(truncate=False)

# Coherencia métrica ↔ unidad
eventos.groupBy("metric", "unit").count().orderBy("metric", "unit").show()

### 7.4 Costos: negativos, percentiles y spikes

In [ ]:
cost = F.col("cost_usd_increment")
eventos.select(
    F.sum((cost < 0).cast("int")).alias("negativos"),
    F.sum((cost < -0.01).cast("int")).alias("menores_a_-0.01"),
    F.min(cost).alias("min"),
    F.max(cost).alias("max"),
).show()

p = eventos.approxQuantile("cost_usd_increment", [0.01, 0.25, 0.5, 0.75, 0.99, 0.999], 0.0001)
print("Percentiles p1, p25, p50, p75, p99, p99.9:", [round(x, 4) for x in p])

### 7.5 Desorden temporal (late data)

In [ ]:
# Si cada archivo cubre casi todo el rango temporal, los eventos no llegan en orden cronológico:
# el streaming necesitará watermark para tolerar datos tardíos.
por_archivo = (eventos.groupBy("source_file")
    .agg(F.min("timestamp").alias("ts_min"), F.max("timestamp").alias("ts_max"))
    .withColumn("dias_cubiertos", F.datediff(F.to_date("ts_max"), F.to_date("ts_min"))))
por_archivo.orderBy("source_file").show(5, truncate=False)
por_archivo.select(F.min("dias_cubiertos").alias("min_dias"), F.avg("dias_cubiertos").alias("prom_dias")).show()

## 8. Chequeos de calidad específicos

### 8.1 Facturación: tipo de cambio por moneda (decisión D-01)

In [ ]:
billing = dfs["billing_monthly"]
fx = F.col("exchange_rate_to_usd").cast("double")
(billing.groupBy("currency")
    .agg(F.count("*").alias("facturas"), F.min(fx).alias("fx_min"), F.max(fx).alias("fx_max"))
    .show())

print("Facturas USD con FX != 1:", billing.filter((F.col("currency") == "USD") & (fx != 1)).count())
print("Subtotales negativos:", billing.filter(F.col("subtotal").cast("double") < 0).count())
print("Créditos nulos:", billing.filter(F.col("credits").isNull()).count())

### 8.2 Rangos de satisfacción: CSAT (1–5) y NPS (−100..100)

In [ ]:
csat = F.col("csat").cast("double")
tickets = dfs["support_tickets"]
tickets.groupBy("csat").count().orderBy("csat").show()
print("CSAT fuera de 1–5:", tickets.filter(csat.isNotNull() & ((csat < 1) | (csat > 5))).count())

for nombre in ["customers_orgs", "nps_surveys"]:
    nps = F.col("nps_score").cast("double")
    dfs[nombre].select(
        F.lit(nombre).alias("fuente"),
        F.min(nps).alias("min"), F.max(nps).alias("max"),
        F.sum(((nps < -100) | (nps > 100)).cast("int")).alias("fuera_de_rango"),
    ).show()

### 8.3 Tickets abiertos y coherencia de fechas

In [ ]:
tickets.select(
    F.sum(F.col("resolved_at").isNull().cast("int")).alias("abiertos"),
    F.sum((F.to_date("resolved_at") < F.to_date("created_at")).cast("int")).alias("resueltos_antes_de_crear"),
    F.sum((F.col("severity") == "critical").cast("int")).alias("criticos"),
    F.sum((F.col("sla_breached") == "True").cast("int")).alias("sla_breach"),
).show()

### 8.4 Integridad referencial de eventos contra maestros

In [ ]:
orgs = dfs["customers_orgs"].select("org_id")
recursos = dfs["resources"].select("resource_id")

print("org_id de eventos sin maestro:", eventos.select("org_id").distinct().join(orgs, "org_id", "left_anti").count())
print("resource_id de eventos sin maestro:", eventos.select("resource_id").distinct().join(recursos, "resource_id", "left_anti").count())

### 8.5 Cobertura temporal entre fuentes

In [ ]:
rangos = [
    ("usage_events",   eventos,                     "timestamp"),
    ("billing",        billing,                     "month"),
    ("tickets",        tickets,                     "created_at"),
    ("nps_surveys",    dfs["nps_surveys"],          "survey_date"),
]
for nombre, df, col in rangos:
    r = df.agg(F.min(F.to_date(col)).alias("desde"), F.max(F.to_date(col)).alias("hasta")).first()
    print(f"{nombre:<14} {r['desde']} → {r['hasta']}")
print("\nRiesgo documentado: la facturación arranca en junio, los eventos el 03/07.")

## 9. Conclusiones

| Hallazgo | Tratamiento previsto |
|---|---|
| `value` inferido como string (mezcla número/texto/null) | Cast con fallback a null en Silver |
| Esquema v1 → v2 desde 2025-07-18 (`carbon_kg`, `genai_tokens`) | Unificación con columnas nulas para v1 |
| `unit` nulo en parte de los eventos | Regla: `unit` no nulo cuando existe `value` → quarantine |
| Costos negativos y spikes (p99 ≪ máx.) | Regla `cost >= -0.01` + flag de anomalía (MAD/percentiles) |
| Eventos desordenados dentro de cada archivo | Watermark + deduplicación por `event_id` en streaming |
| FX de USD ≠ 1 | D-01: forzar FX = 1 + `fx_corrected_flag` |
| CSAT y NPS fuera de rango | Validación de rango → null + flag |
| Facturación desde junio, eventos desde julio | Supuesto documentado; conciliación solo jul–ago |

Detalle completo en `docs/02_5v_e_inventario_fuentes.md`.

In [ ]:
# Liberar recursos al terminar
# spark.stop()